In [ ]:
# Solar Smart Charger Project
# Cell 1: Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

print("Solar Smart Charger Dashboard")
print("Libraries loaded successfully!")

Solar Smart Charger Dashboard
Libraries loaded successfully!


In [ ]:
# Cell 2: Sample experimental dataset

data = pd.DataFrame({
    "Sunlight": ["Low", "Moderate", "Good", "Strong", "Very Strong"],
    "Solar_Voltage_V": [4.0, 4.8, 5.2, 5.7, 6.0],
    "Solar_Current_A": [0.15, 0.30, 0.45, 0.65, 0.75],
    "Output_Voltage_V": [5.0, 5.0, 5.0, 5.0, 5.0],
    "Output_Current_A": [0.10, 0.20, 0.30, 0.40, 0.45],
    "Charging_Time_Min": [180, 120, 85, 65, 55]
})

data["Solar_Power_W"] = (
    data["Solar_Voltage_V"] * data["Solar_Current_A"]
)

data["Output_Power_W"] = (
    data["Output_Voltage_V"] * data["Output_Current_A"]
)

data["Efficiency_Percent"] = (
    data["Output_Power_W"] / data["Solar_Power_W"] * 100
).round(2)

display(data)

,Sunlight,Solar_Voltage_V,Solar_Current_A,Output_Voltage_V,Output_Current_A,Charging_Time_Min,Solar_Power_W,Output_Power_W,Efficiency_Percent
0,Low,4.0,0.15,5.0,0.10,180,0.600,0.50,83.33
1,Moderate,4.8,0.30,5.0,0.20,120,1.440,1.00,69.44
2,Good,5.2,0.45,5.0,0.30,85,2.340,1.50,64.10
3,Strong,5.7,0.65,5.0,0.40,65,3.705,2.00,53.98
4,Very Strong,6.0,0.75,5.0,0.45,55,4.500,2.25,50.00


In [1]:

# CELL 3: SOLAR SMART CHARGER DASHBOARD

import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
from google.colab import files

# --------------------------------------------------
# 1. INPUT CONTROLS
# --------------------------------------------------

sunlight = widgets.FloatSlider(
    value=5.0, min=0.5, max=6.0, step=0.1,
    description="Solar V:",
    continuous_update=False
)

solar_current = widgets.FloatSlider(
    value=0.5, min=0.0, max=0.83, step=0.01,
    description="Solar A:",
    continuous_update=False
)

output_current = widgets.FloatSlider(
    value=0.30, min=0.0, max=2.0, step=0.05,
    description="USB A:",
    continuous_update=False
)

battery_capacity = widgets.IntSlider(
    value=5000, min=1000, max=10000, step=500,
    description="Battery mAh:",
    continuous_update=False
)

battery_level = widgets.IntSlider(
    value=40, min=0, max=100, step=5,
    description="Battery %:",
    continuous_update=False
)

run_button = widgets.Button(
    description="Analyze Performance",
    button_style="success",
    icon="bar-chart"
)

export_button = widgets.Button(
    description="Export Results CSV",
    button_style="info",
    icon="download"
)

output_area = widgets.Output()
latest_result = {}

# --------------------------------------------------
# 2. ANALYSIS FUNCTION
# --------------------------------------------------

def analyze_performance(_=None):
    global latest_result

    # Read current slider values
    V_solar = sunlight.value
    I_solar = solar_current.value
    V_usb = 5.0
    I_usb = output_current.value

    capacity = battery_capacity.value
    level = battery_level.value

    # Calculate input and output power
    solar_power = V_solar * I_solar
    usb_power = V_usb * I_usb

    # Instantaneous power ratio, NOT full system efficiency
    efficiency = (
        usb_power / solar_power * 100
        if solar_power > 0 else 0
    )

    # Approximate charging time
    effective_charge_current = I_solar * 1000 * 0.85
    remaining_capacity = capacity * (100 - level) / 100

    if effective_charge_current > 0:
        charging_time = (
            remaining_capacity / effective_charge_current
        )
    else:
        charging_time = None

    # System status
    if solar_power == 0:
        status = "No solar power available"
    elif usb_power > solar_power:
        status = "Warning: output exceeds solar input"
    elif V_solar < 4.5:
        status = "Low solar voltage - charging may be limited"
    else:
        status = "Solar input available - check hardware readings"

    # Store results for CSV export
    latest_result = {
        "Solar_Voltage_V": V_solar,
        "Solar_Current_A": I_solar,
        "Solar_Input_Power_W": solar_power,
        "USB_Voltage_V": V_usb,
        "USB_Current_A": I_usb,
        "USB_Output_Power_W": usb_power,
        "Power_Ratio_Percent": efficiency,
        "Battery_Capacity_mAh": capacity,
        "Battery_Level_Percent": level,
        "Estimated_Charging_Time_Hours": charging_time,
        "System_Status": status
    }

    # --------------------------------------------------
    # 3. DISPLAY RESULTS AND GRAPHS
    # --------------------------------------------------

    with output_area:
        clear_output(wait=True)

        print("=" * 48)
        print("     SOLAR SMART CHARGER DASHBOARD")
        print("=" * 48)

        print(f"\nSolar Voltage       : {V_solar:.2f} V")
        print(f"Solar Current       : {I_solar:.2f} A")
        print(f"Solar Input Power   : {solar_power:.2f} W")

        print(f"\nUSB Output Voltage  : {V_usb:.2f} V")
        print(f"USB Output Current  : {I_usb:.2f} A")
        print(f"USB Output Power    : {usb_power:.2f} W")

        print(f"\nPower Ratio         : {efficiency:.2f}%")
        print(f"Battery Capacity    : {capacity} mAh")
        print(f"Battery Charge Level: {level}%")

        if charging_time is not None:
            print(f"Estimated Charging Time: {charging_time:.2f} hours")
        else:
            print("Estimated Charging Time: Cannot estimate")

        print(f"\nSystem Status: {status}")

        if efficiency > 100:
            print("WARNING: Check input/output readings.")
            print("The battery may also be supplying USB power.")

        # GRAPH 1: Current input versus output power
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=120)

        axes[0].bar(
            ["Solar Input", "USB Output"],
            [solar_power, usb_power]
        )
        axes[0].set_title("Solar Input vs USB Output Power")
        axes[0].set_ylabel("Power (W)")
        axes[0].grid(axis="y", alpha=0.3)

        # GRAPH 2: Efficiency ratio under simulated sunlight
        conditions = [
            "Low", "Moderate", "Good", "Strong", "Very Strong"
        ]

        # Simulated voltage and current at different sunlight levels
        voltage_factors = [0.67, 0.80, 0.87, 0.95, 1.00]
        current_factors = [0.20, 0.40, 0.60, 0.80, 1.00]

        efficiency_values = []

        for vf, cf in zip(voltage_factors, current_factors):
            simulated_voltage = V_solar * vf
            simulated_current = I_solar * cf
            simulated_input_power = (
                simulated_voltage * simulated_current
            )

            if simulated_input_power > 0:
                ratio = (
                    usb_power / simulated_input_power
                ) * 100
            else:
                ratio = 0

            efficiency_values.append(ratio)

        axes[1].plot(
            conditions,
            efficiency_values,
            marker="o"
        )
        axes[1].set_title("Power Ratio by Simulated Sunlight")
        axes[1].set_ylabel("Output/Input Power (%)")
        axes[1].set_xlabel("Simulated sunlight condition")
        axes[1].tick_params(axis="x", rotation=30)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        print("\nNote: Inputs and sunlight conditions are simulated.")
        print("Charging time is an approximate estimate.")
        print("The second graph is a power ratio, not measured efficiency.")

# --------------------------------------------------
# 4. CSV EXPORT
# --------------------------------------------------

def export_results(_):
    if not latest_result:
        with output_area:
            print("Click Analyze Performance first.")
        return

    filename = "solar_charger_results.csv"

    pd.DataFrame([latest_result]).to_csv(
        filename, index=False
    )

    files.download(filename)

# --------------------------------------------------
# 5. BUTTON ACTIONS
# --------------------------------------------------

run_button.on_click(analyze_performance)
export_button.on_click(export_results)

# --------------------------------------------------
# 6. DISPLAY DASHBOARD
# --------------------------------------------------

display(widgets.VBox([
    widgets.HTML(
        "<h2>Solar Smart Charger Monitoring System</h2>"
    ),
    widgets.HTML(
        "<p>Adjust the inputs, then click Analyze Performance.</p>"
    ),
    sunlight,
    solar_current,
    output_current,
    battery_capacity,
    battery_level,
    widgets.HBox([run_button, export_button]),
    output_area
]))

# Display initial results
analyze_performance()